In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, trim, lower, current_timestamp
import uuid

tabela_bronze = "credit_risk.bronze.bronze_credit_risk"
tabela_silver = "credit_risk.silver.silver_credit_risk"
execution_id = str(uuid.uuid4())

try:
    df_bronze = spark.read.table(tabela_bronze)
    
    for c in df_bronze.columns:
        df_bronze = df_bronze.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
        
    if "unnamed:_0" in df_bronze.columns:
        df_bronze = df_bronze.drop("unnamed:_0")
        
    df_silver = df_bronze \
        .fillna("desconhecido", subset=["saving_accounts", "checking_account"]) \
        .withColumn("purpose", trim(lower(col("purpose")))) \
        .withColumn("housing", trim(lower(col("housing")))) \
        .withColumn("sex", trim(lower(col("sex"))))

    df_silver.write.format("delta").mode("overwrite").saveAsTable(tabela_silver)
    
    spark.sql(f"""
        INSERT INTO credit_risk.bronze.log_pipeline_execution 
        VALUES ('{execution_id}', '02_silver_cleaning', '{tabela_silver}', {df_silver.count()}, 'SUCCESS', '', current_timestamp())
    """)
    print("Processamento Silver concluído.")

except Exception as e:
    erro = str(e).replace("'", "")
    spark.sql(f"""
        INSERT INTO credit_risk.bronze.log_pipeline_execution 
        VALUES ('{execution_id}', '02_silver_cleaning', '{tabela_silver}', 0, 'FAILED', '{erro}', current_timestamp())
    """)
    raise e